# Midi Audio Synthesis

The .mid file were available at https://zenodo.org/records/15470412 ([github](https://github.com/MTG/SymbTr)). We will use the `midi2audio` library to synthesize the .mid files into .wav files. The `midi2audio` library is a Python wrapper for the FluidSynth software synthesizer, which can render MIDI files using SoundFont files.

The soundfont files were available to be downloaded from community site (https://www.keyfimuzik.net/fl-studio/106943-turkish-instruments-soundfont-sf2-dev-arsiv.html) (big thanks to [Dmex35](https://www.keyfimuzik.net/members/363067-dmex35.html)) and we have added the metadata of the soundfont files in `data/soundfonts_programs.json`. We will use these soundfont files to synthesize the MIDI files into audio.

In [ ]:
from midi2audio import FluidSynth
import os
import sys
from contextlib import contextmanager, redirect_stderr, redirect_stdout

@contextmanager
def suppress_stdout_stderr():
    """A context manager that redirects stdout and stderr to devnull"""
    with open(os.devnull, 'w') as fnull:
        with redirect_stderr(fnull), redirect_stdout(fnull):
            yield

def synthesize_turkish_midi(midi_path, soundfont_path, output_wav_path):
    """
    Renders a microtonal SymbTr MIDI file into a .wav file using a specific SoundFont.
    """
    # Initialize FluidSynth with your downloaded Turkish .sf2 SoundFont
    fs = FluidSynth(sound_font=soundfont_path)

    # Check if files exist to prevent errors
    if not os.path.exists(midi_path):
        print(f"Error: MIDI file not found at {midi_path}")
        return
    if not os.path.exists(soundfont_path):
        print(f"Error: SoundFont not found at {soundfont_path}")
        return

    # Render the audio stem
    print(f"Synthesizing {midi_path}...")

    # Wrap the call to hide warnings/logs
    with suppress_stdout_stderr():
        fs.midi_to_audio(midi_path, output_wav_path)

    print(f"Success! Audio saved to {output_wav_path}")


In [5]:
import mido

def inspect_midi_data(midi_path):
    """
    Loads a MIDI file and prints its internal messages,
    allowing you to see the baked-in microtonal pitch bends.
    """
    # Load the MIDI file
    mid = mido.MidiFile(midi_path)

    print(f"Inspecting: {midi_path}")
    print(f"Total Playback Time: {mid.length:.2f} seconds")

    # Iterate through the tracks and messages
    for i, track in enumerate(mid.tracks):
        print(f"\n--- Track {i}: {track.name} ---")

        # Print the first 15 messages just to see the structure
        for msg in track[:15]:
            # This is where you will see 'note_on', 'note_off', and crucial 'pitchwheel' events
            print(msg)

Upon testing and inspection however, the midi produces silence. It turns out that the SymbTR MIDI file is explicitly commanding the synthesizer to play Program 0 (a one of the "slots" in standard MIDI file structure, and 0 is universally reserved for the Acoustic Grand Piano), while many independent creators of custom Turkish SoundFonts do not map their instruments to Program 0.

To address this, we inspected the .sf2 files with open-source program, [Polyphone](https://www.polyphone.io/), and found the correct program numbers for the instruments we need, hence the "program" key in the `data/soundfonts_programs.json` file. After this fix, we were able to successfully synthesize the MIDI files into audio with the correct instruments!

In [17]:
def fix_midi_program(input_path, output_path, correct_program_number):
    """
    Overwrites the instrument program number in a MIDI file
    to match a custom SoundFont.
    """
    # Load the original SymbTr MIDI file
    mid = mido.MidiFile(input_path)

    # Iterate through all tracks and messages
    for track in mid.tracks:
        for msg in track:
            # When you find the instrument selection message, change it
            if msg.type == 'program_change':
                msg.program = correct_program_number

    # Save the corrected file
    mid.save(output_path)
    # print(f"Fixed MIDI saved to: {output_path}")


In [ ]:
# midi_file = "../data/midis/acem--ilahi--duyek--aldanma_dunya--zekai_dede.mid"         # Your SymbTr MIDI file
midi_file = "../data/midis/fixed/fixed_symbtr_file.mid"         # Your SymbTr MIDI file
sf2_file = "../data/soundfonts/AngaraSazDuz.sf2"       # Your downloaded SoundFont
output_file = "../data/synthesized/acem--ilahi--duyek--aldanma_dunya--zekai_dede.wav"       # The resulting audio stem

In [5]:
inspect_midi_data(midi_file)

Inspecting: ../data/midis/acem--ilahi--duyek--aldanma_dunya--zekai_dede.mid
Total Playback Time: 65.32 seconds

--- Track 0:  ---
MetaMessage('copyright', text='(C) 2025, Mus2-Alpha/SymbTr', time=0)
MetaMessage('time_signature', numerator=8, denominator=8, clocks_per_click=12, notated_32nd_notes_per_beat=8, time=0)
MetaMessage('set_tempo', tempo=714286, time=0)
MetaMessage('end_of_track', time=0)

--- Track 1:  ---
control_change channel=0 control=101 value=0 time=0
control_change channel=0 control=100 value=0 time=0
control_change channel=0 control=6 value=0 time=0
control_change channel=0 control=38 value=0 time=0
program_change channel=0 program=0 time=0
pitchwheel channel=0 pitch=-155 time=0
note_on channel=0 note=72 velocity=96 time=0
note_off channel=0 note=72 velocity=96 time=326
pitchwheel channel=0 pitch=-155 time=17
note_on channel=0 note=0 velocity=0 time=0
note_off channel=0 note=0 velocity=0 time=0
pitchwheel channel=0 pitch=-232 time=0
note_on channel=0 note=77 velocity=9

In [ ]:
# Example Usage: Replace 25 with the actual number you found in Polyphone
fix_midi_program(midi_file, "../data/midis/fixed/fixed_symbtr_file.mid", 1)

In [ ]:
synthesize_turkish_midi(midi_file, sf2_file, output_file)

Synthesizing ../data/midis/fixed/fixed_symbtr_file.mid...


fluidsynth: warning: No preset found on channel 0 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 1 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 2 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 3 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 4 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 5 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 6 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 7 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 8 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 9 [bank=128 prog=0]
fluidsynth: warning: No preset found on channel 10 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 11 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 12 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 13 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 14 [bank=0 prog=0]
flu

FluidSynth runtime version 2.3.4
Copyright (C) 2000-2023 Peter Hanappe and others.
Distributed under the LGPL license.
SoundFont(R) is a registered trademark of Creative Technology Ltd.

Rendering audio to file '../data/synthesized/acem--ilahi--duyek--aldanma_dunya--zekai_dede.wav'..
Success! Audio saved to ../data/synthesized/acem--ilahi--duyek--aldanma_dunya--zekai_dede.wav


## Batch Processing

In [7]:
DATA_DIR = "../data"

In [12]:
# read json file to see the soundfont programs
import json

with open(os.path.join(DATA_DIR, "soundfonts_programs.json"), "r") as f:
    soundfont_programs = json.load(f)

soundfont_programs

{'soundfonts': [{'name': 'baglama_saz',
   'filename': 'AngaraSazDuz.sf2',
   'description': 'Bağlama/saz soundfont (Yamaha), the Plucked Lute (Metal Strings) class',
   'program': '001'},
  {'name': 'kanun',
   'filename': 'Kanun Duz.sf2',
   'description': 'Kanun soundfont (Yamaha), the Plucked Zither class',
   'program': '001'},
  {'name': 'ney',
   'filename': 'Ney1.sf2',
   'description': 'Ney soundfont (Yamaha), the Woodwind (End-blown Flute) class',
   'program': '001'},
  {'name': 'kemence',
   'filename': 'Kemence 1.sf2',
   'description': 'Kemençe soundfont (Yamaha), the Bowed Strings class',
   'program': '001'},
  {'name': 'ud',
   'filename': 'uD Oud.sf2',
   'description': 'Ud (Oud) soundfont (Yamaha), The Plucked Lute (Nylon/Gut Strings) class',
   'program': '033'},
  {'name': 'zurna',
   'filename': 'Zurna Duz 2.sf2',
   'description': 'Zurna soundfont (Yamaha), The Double-Reed (Aerophone) class',
   'program': '001'}],
 'source': 'https://www.keyfimuzik.net/fl-studio

In [18]:
from tqdm.auto import tqdm

ori_midi_dir = os.path.join(DATA_DIR, "midis", "symbtr", "mid_v3")

for sf in soundfont_programs["soundfonts"]:
    sf_midi_dir = os.path.join(DATA_DIR, "midis", "fixed", sf["name"])

    if not os.path.exists(sf_midi_dir):
        os.makedirs(sf_midi_dir, exist_ok=True)

    # fix the original midi file for this soundfont
    for midi in tqdm(os.listdir(ori_midi_dir), desc=f"Processing {sf['name']}"):
        input_midi_path = os.path.join(ori_midi_dir, midi)
        output_midi_path = os.path.join(sf_midi_dir, midi)
        fix_midi_program(input_midi_path, output_midi_path, int(sf["program"]))

Processing zurna: 100%|██████████| 3000/3000 [00:41<00:00, 71.45it/s]


In [20]:
LIMITS = 5 # set to None to synthesize all the midi files

for sf in soundfont_programs["soundfonts"]:
    for midi in tqdm(os.listdir(os.path.join(DATA_DIR, "midis", "fixed", sf["name"]))[:LIMITS], desc=f"Synthesizing {sf['name']}"):
        # synthesize the fixed midi file with the corresponding soundfont
        if not os.path.exists(os.path.join(DATA_DIR, "synthesized", sf["name"])):
            os.makedirs(os.path.join(DATA_DIR, "synthesized", sf["name"]), exist_ok=True)
        output_wav_path = os.path.join(DATA_DIR, "synthesized", sf["name"], f"{midi[:-4]}.wav")

        synthesize_turkish_midi(output_midi_path,
                 os.path.join(DATA_DIR, "soundfonts", sf["filename"]),
                                output_wav_path)

Synthesizing baglama_saz:   0%|          | 0/5 [00:00<?, ?it/s]fluidsynth: warning: No preset found on channel 0 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 1 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 2 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 3 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 4 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 5 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 6 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 7 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 8 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 9 [bank=128 prog=0]
fluidsynth: warning: No preset found on channel 10 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 11 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 12 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 13 [bank=0 prog=0]
fluidsy

Synthesizing ../data/midis/fixed/zurna/sehnaz--sazsemaisi--aksaksemai----arif_sami_toker.mid...


Synthesizing baglama_saz:  20%|██        | 1/5 [00:05<00:23,  5.80s/it]

FluidSynth runtime version 2.3.4
Copyright (C) 2000-2023 Peter Hanappe and others.
Distributed under the LGPL license.
SoundFont(R) is a registered trademark of Creative Technology Ltd.

Rendering audio to file '../data/synthesized/baglama_saz/segah--pesrev--devrikebir----cihat_hircin.wav'..
Success! Audio saved to ../data/synthesized/baglama_saz/segah--pesrev--devrikebir----cihat_hircin.wav
Synthesizing ../data/midis/fixed/zurna/sehnaz--sazsemaisi--aksaksemai----arif_sami_toker.mid...


fluidsynth: warning: No preset found on channel 0 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 1 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 2 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 3 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 4 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 5 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 6 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 7 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 8 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 9 [bank=128 prog=0]
fluidsynth: warning: No preset found on channel 10 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 11 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 12 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 13 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 14 [bank=0 prog=0]
flu

...

FluidSynth runtime version 2.3.4
Copyright (C) 2000-2023 Peter Hanappe and others.
Distributed under the LGPL license.
SoundFont(R) is a registered trademark of Creative Technology Ltd.

Rendering audio to file '../data/synthesized/zurna/hicaz--sarki--evsat--nicin_a_sevdigim--nikogos_aga.wav'..
Success! Audio saved to ../data/synthesized/zurna/hicaz--sarki--evsat--nicin_a_sevdigim--nikogos_aga.wav
Synthesizing ../data/midis/fixed/zurna/sehnaz--sazsemaisi--aksaksemai----arif_sami_toker.mid...


fluidsynth: warning: No preset found on channel 0 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 1 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 2 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 3 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 4 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 5 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 6 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 7 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 8 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 9 [bank=128 prog=0]
fluidsynth: warning: No preset found on channel 10 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 11 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 12 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 13 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 14 [bank=0 prog=0]
flu

FluidSynth runtime version 2.3.4
Copyright (C) 2000-2023 Peter Hanappe and others.
Distributed under the LGPL license.
SoundFont(R) is a registered trademark of Creative Technology Ltd.

Rendering audio to file '../data/synthesized/zurna/hicaz--zeybek--aksak--izmir_zeybegi--izmir.wav'..
Success! Audio saved to ../data/synthesized/zurna/hicaz--zeybek--aksak--izmir_zeybegi--izmir.wav
Synthesizing ../data/midis/fixed/zurna/sehnaz--sazsemaisi--aksaksemai----arif_sami_toker.mid...


fluidsynth: warning: No preset found on channel 0 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 1 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 2 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 3 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 4 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 5 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 6 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 7 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 8 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 9 [bank=128 prog=0]
fluidsynth: warning: No preset found on channel 10 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 11 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 12 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 13 [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 14 [bank=0 prog=0]
flu

FluidSynth runtime version 2.3.4
Copyright (C) 2000-2023 Peter Hanappe and others.
Distributed under the LGPL license.
SoundFont(R) is a registered trademark of Creative Technology Ltd.

Rendering audio to file '../data/synthesized/zurna/neva--murabba--muhammes--zeyn_eden--dede_efendi.wav'..
Success! Audio saved to ../data/synthesized/zurna/neva--murabba--muhammes--zeyn_eden--dede_efendi.wav
